# 📦 LangGraph-based WikiLLM ReAct PARA Vault Organizer Agent

Welcome to the interactive notebook for the **LangGraph WikiLLM PARA Vault Organizer Agent**. This notebook breaks down the monolithic `vault_agent.py` orchestrator script into step-by-step, executable Python cells, providing a deep-dive educational analysis of how **StateGraph, Nodes, and Edges** are used to orchestrate this production-grade pipeline.

## 🗺️ LangGraph Architectural Workflow Diagram

The agent uses a StateGraph to coordinate each step of the note organization workflow. State is passed between nodes, allowing safe transactions and telemetry recording.

```mermaid
graph TD
    Start([Bắt đầu Notebook]) --> git_sandbox_init[1. node_git_sandbox_init<br><i>git stash -u & checkout sandbox branch</i>]
    git_sandbox_init --> scan_directory[2. node_scan_directory<br><i>Call organize.py scan to discover notes</i>]
    scan_directory --> semantic_classify[3. node_semantic_classify<br><i>AI batch categorization & flattening path gate</i>]
    semantic_classify --> generate_move_plan[4. node_generate_move_plan<br><i>Call organize.py plan</i>]
    generate_move_plan --> execute_migration[5. node_execute_migration<br><i>Call organize.py execute to move notes & repair links</i>]
    execute_migration --> review_gate[6. node_review_gate<br><i>Prompt user: merge/approve or abort/reject</i>]
    review_gate --> git_sandbox_resolve[7. node_git_sandbox_resolve<br><i>Switch back to original branch, resolve Git</i>]
    git_sandbox_resolve --> END([END])
```

---

## 🛠️ Step 0: Imports, Environment Loader & State Dictionary

We import the standard subsystems and define our `OrganizerState` which inherits from `TypedDict`. This is the single source of truth (SSoT) state shared across all nodes in the LangGraph workflow.

In [ ]:
import os
import sys
import re
import json
import time
import subprocess
from datetime import datetime, timezone
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional, TypedDict

# Import LangGraph
try:
    from langgraph.graph import StateGraph, END
except ImportError:
    log_warning("Installing langgraph and langchain-core...")
    subprocess.run([sys.executable, "-m", "pip", "install", "langgraph", "langchain-core", "openai", "python-dotenv"])
    from langgraph.graph import StateGraph, END

# Add scripts directory to sys.path so sibling imports work
NOTEBOOK_DIR = Path(os.getcwd())
SCRIPTS_DIR = NOTEBOOK_DIR / "vault-organizer-agent" / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

# Console colors for formatted outputs
class Colors:
    HEADER = '\033[95m'
    OKBLUE = '\033[94m'
    OKGREEN = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'

def log_header(msg: str):
    print(f"\n{Colors.HEADER}{Colors.BOLD}=== {msg} ==={Colors.ENDC}")

def log_info(msg: str):
    print(f"{Colors.OKBLUE}[INFO] {msg}{Colors.ENDC}")

def log_success(msg: str):
    print(f"{Colors.OKGREEN}[SUCCESS] {msg}{Colors.ENDC}")

def log_warning(msg: str):
    print(f"{Colors.WARNING}[WARNING] {msg}{Colors.ENDC}")

def log_error(msg: str):
    print(f"{Colors.FAIL}[ERROR] {msg}{Colors.ENDC}")

def find_and_load_env() -> bool:
    """Walk up from notebook working dir to locate .env file."""
    current = NOTEBOOK_DIR
    for _ in range(5):
        env_path = current / ".env"
        if env_path.is_file():
            try:
                from dotenv import load_dotenv
                load_dotenv(env_path)
                log_success(f"Loaded environment from: {env_path}")
                return True
            except ImportError:
                log_warning("dotenv package missing. Parsing environment variables natively.")
                with open(env_path, "r", encoding="utf-8") as f:
                    for line in f:
                        line = line.strip()
                        if line and not line.startswith("#") and "=" in line:
                            k, v = line.split("=", 1)
                            os.environ[k.strip()] = v.strip().strip("'\"")
                return True
        parent = current.parent
        if parent == current:
            break
        current = parent
    log_warning("No .env file found walking up directories.")
    return False

find_and_load_env()

class OrganizerState(TypedDict):
    vault_root: str
    original_branch: str
    sandbox_branch: str
    stashed: bool
    scan_data: Dict[str, Any]
    ai_classified: List[Dict[str, Any]]
    execute_summary: Dict[str, Any]
    user_decision: str

: 

## 🛡️ Step 1: LLM Connector & SSoT Path Sanitization

We implement the `LLMClassifier` to safely execute semantic classifications under the system promt persona defined in `SUB-SKILLS.md`. We also establish the `sanitize_category_gate` which caps taxonomy depth under `3_RESOURCES/` to a single level deep to keep Obsidian flat and easy to search.

In [ ]:
def sanitize_category_gate(category: str) -> str:
    """
    Enforces SSoT Path rules:
    1. Keeps category alphanumeric, spaces, and hyphens.
    2. Capping taxonomy depth under 3_RESOURCES/ to exactly 1 level.
    """
    category = category.strip().replace("\\", "/")
    parts = category.split("/")
    root_folder = parts[0]
    
    valid_para = {"1_PROJECTS", "2_ACTIONS", "2_AREAS", "3_RESOURCES", "4_ARCHIVES"}
    if root_folder not in valid_para:
        root_folder = "3_RESOURCES"
        if len(parts) > 1:
            category = f"3_RESOURCES/{'/'.join(parts[1:])}"
        else:
            category = f"3_RESOURCES/{category}"
            parts = category.split("/")
            
    if root_folder == "3_RESOURCES" and len(parts) > 2:
        flattened_sub = parts[-1]
        log_info(f"SSoT Depth Gate: Flattening resource path '{category}' to '3_RESOURCES/{flattened_sub}'")
        category = f"3_RESOURCES/{flattened_sub}"

    sanitized_parts = []
    for part in category.split("/"):
        cleaned = re.sub(r"[^\w\s\-]", "", part).strip()
        if cleaned not in valid_para:
            cleaned = cleaned.title()
        if cleaned:
            sanitized_parts.append(cleaned)
            
    return "/".join(sanitized_parts)

class LLMClassifier:
    def __init__(self, provider: str = "openai", model_name: str = "gpt-4o"):
        self.provider = provider
        self.model_name = model_name
        self.openai_key = os.environ.get("OPENAI_API_KEY")
        self.gemini_key = os.environ.get("GEMINI_API_KEY")

        if not self.openai_key and self.gemini_key:
            self.provider = "google"
            self.model_name = "gemini-1.5-pro"
        elif not self.gemini_key and self.openai_key:
            self.provider = "openai"
            self.model_name = "gpt-4o"

    def query(self, system_prompt: str, user_prompt: str) -> str:
        """Call the selected LLM provider securely."""
        if self.provider == "openai" or self.provider == "local":
            return self._query_openai(system_prompt, user_prompt)
        elif self.provider == "google":
            return self._query_gemini(system_prompt, user_prompt)
        else:
            raise ValueError(f"Unknown LLM provider: {self.provider}")

    def _query_openai(self, system_prompt: str, user_prompt: str) -> str:
        from openai import OpenAI
        base_url = "http://localhost:8000/v1" if self.provider == "local" else None
        client = OpenAI(api_key=self.openai_key or "local-key", base_url=base_url)
        res = client.chat.completions.create(
            model=self.model_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.1,
            response_format={"type": "json_object"} if self.provider == "openai" else None
        )
        return res.choices[0].message.content

    def _query_gemini(self, system_prompt: str, user_prompt: str) -> str:
        import google.generativeai as genai
        genai.configure(api_key=self.gemini_key)
        model = genai.GenerativeModel(
            model_name=self.model_name,
            generation_config={"response_mime_type": "application/json"},
            system_instruction=system_prompt
        )
        res = model.generate_content(user_prompt)
        return res.text

## 🏷️ Step 2: LangGraph Node Definitions

Here we define our individual nodes which execute modular steps, inspect the state, modify it, and return the modified state payload back to the LangGraph executor.

In [ ]:
def node_git_sandbox_init(state: OrganizerState) -> Dict[str, Any]:
    log_header("Phase 1: Git Sandboxing & Pre-flight")
    vault_root = Path(state["vault_root"])

    # 1. Capture Original Branch
    res = subprocess.run(
        ["git", "rev-parse", "--abbrev-ref", "HEAD"],
        cwd=str(vault_root), check=True,
        capture_output=True, text=True
    )
    original_branch = res.stdout.strip()
    log_info(f"Original branch captured: {original_branch}")

    # 2. Check for Dirty Tree & Auto-Stash
    status_res = subprocess.run(
        ["git", "status", "--porcelain"],
        cwd=str(vault_root), check=True,
        capture_output=True, text=True
    )
    stashed = False
    if status_res.stdout.strip():
        log_warning("Workspace is dirty. Stashing active changes to clean directory...")
        subprocess.run(
            ["git", "stash", "save", "vault-organizer-agent: Active pre-organize workspace snapshot", "-u"],
            cwd=str(vault_root), check=True,
            capture_output=True, text=True
        )
        stashed = True
        log_success("Active drafts stashed successfully.")

    # 3. Create & Checkout Sandbox Branch
    timestamp = datetime.now().strftime("%Y-%m-%d-%H%M%S")
    sandbox_branch = f"vault-organize/sandbox-{timestamp}"
    log_info(f"Checking out sandbox branch: {sandbox_branch}...")
    subprocess.run(
        ["git", "checkout", "-b", sandbox_branch],
        cwd=str(vault_root), check=True,
            capture_output=True, text=True
    )
    log_success(f"Sandbox checked out cleanly.")

    return {
        "original_branch": original_branch,
        "sandbox_branch": sandbox_branch,
        "stashed": stashed
    }

def node_scan_directory(state: OrganizerState) -> Dict[str, Any]:
    log_header("Phase 2: Directory Scanning")
    vault_root = Path(state["vault_root"])

    log_info("Running directory scanner tool...")
    scan_process = subprocess.run(
        [sys.executable, str(SCRIPTS_DIR / "organize.py"), "scan"],
        cwd=str(vault_root), capture_output=True, text=True, check=True
    )
    scan_data = json.loads(scan_process.stdout)
    
    log_success(f"Scanning completed. Files found: {scan_data.get('total_unorganized', 0)}")
    return {"scan_data": scan_data}

def node_semantic_classify(state: OrganizerState) -> Dict[str, Any]:
    scan_data = state["scan_data"]
    needs_ai = scan_data.get("needs_ai", [])
    heuristic_matched = scan_data.get("heuristic_matched", [])
    existing_categories = scan_data.get("existing_categories", [])
    
    ai_classified = []
    if needs_ai:
        log_header("Phase 3: AI Semantic Classification")
        
        # Load SUB-SKILLS prompt
        sub_skills_path = NOTEBOOK_DIR / "vault-organizer-agent" / "SUB-SKILLS.md"
        if not sub_skills_path.exists():
            raise FileNotFoundError(f"Sub-skills prompt file missing: {sub_skills_path}")
        with open(sub_skills_path, "r", encoding="utf-8") as f:
            system_prompt = f.read()

        # Dynamic Focus Slot Ingestion
        extensions = [Path(f["filename"]).suffix.lower() for f in needs_ai]
        is_code_batch = sum(1 for ext in extensions if ext in (".py", ".js", ".ts", ".rs", ".go", ".c", ".cpp")) >= len(needs_ai) * 0.4
        
        dynamic_focus = "Focus strictly on the core conceptual thesis, systems relationships, and actionable deliverables."
        if is_code_batch:
            dynamic_focus = "Focus strictly on programming language syntax, libraries used, object-oriented class structures, and algorithm complexity."
            log_info("Dynamic Focus detected: Coding/Engineering Batch.")
        
        system_prompt = system_prompt.replace("[DYNAMIC_TOPIC_FOCUS]", dynamic_focus)
        
        # Setup LLM
        provider = os.environ.get("DEFAULT_PROVIDER", "openai")
        model_name = os.environ.get("DEFAULT_MODEL", "gpt-4o")
        classifier = LLMClassifier(provider=provider, model_name=model_name)
        log_info(f"Spawning AI Classifier ({classifier.provider}:{classifier.model_name})...")
        
        batch_payload = []
        for file_entry in needs_ai:
            batch_payload.append({
                "filename": file_entry["filename"],
                "relative_path": file_entry["relative_path"],
                "content_preview": file_entry.get("content_preview", "")[:1800]
            })

        user_prompt = f"""\n        Analyze and classify the following files into the Obsidian PARA structure (1_PROJECTS, 2_ACTIONS, 3_RESOURCES, 4_ARCHIVES).\n        Existing categories you can choose to align with: {existing_categories}\n\n        Files to classify:\n        {json.dumps(batch_payload, indent=2, ensure_ascii=False)}\n\n        Return strictly a JSON array of objects conforming to the SUB-SKILLS.md schema:\n        [\n          {{\n            "filename": "...",\n            "relative_path": "...",\n            "category": "...",\n            "summary": "...",\n            "confidence": "high"\n          }}\n        ]\n        """
        
        ai_res_text = classifier.query(system_prompt, user_prompt)
        
        ai_res_text = ai_res_text.strip()
        if ai_res_text.startswith("```json"):
            ai_res_text = ai_res_text[7:]
        elif ai_res_text.startswith("```"):
            ai_res_text = ai_res_text[3:]
        if ai_res_text.endswith("```"):
            ai_res_text = ai_res_text[:-3]
        ai_res_text = ai_res_text.strip()
        
        classified_results = json.loads(ai_res_text)
        
        # Enforce SSoT Sanitization Gate
        for entry in classified_results:
            raw_cat = entry.get("category", "_unsorted")
            sanitized_cat = sanitize_category_gate(raw_cat)
            entry["category"] = sanitized_cat
            
            if entry.get("confidence", "low") == "low":
                entry["category"] = "_unsorted"
            
            ai_classified.append(entry)
            log_success(f"AI Categorized: {entry['filename']} -> {entry['category']} ('{entry['summary']}')")

    return {"ai_classified": ai_classified}

def node_generate_move_plan(state: OrganizerState) -> Dict[str, Any]:
    log_header("Phase 4: Planning & Move Document Compilation")
    vault_root = Path(state["vault_root"])
    scan_data = state["scan_data"]
    ai_classified = state["ai_classified"]

    combined_classifications = {
        "vault_root": str(vault_root),
        "ai_root": scan_data.get("ai_root"),
        "heuristic_matched": scan_data.get("heuristic_matched", []),
        "ai_classified": ai_classified
    }
    
    temp_json_path = NOTEBOOK_DIR / "temp_classifications.json"
    with open(temp_json_path, "w", encoding="utf-8") as f:
        json.dump(combined_classifications, f, indent=2, ensure_ascii=False)
        
    log_info("Compiling move_plan.md and _move_plan.json...")
    subprocess.run(
        [sys.executable, str(SCRIPTS_DIR / "organize.py"), "plan", "-i", str(temp_json_path)],
        cwd=str(vault_root), capture_output=True, text=True, check=True
    )
    
    if temp_json_path.exists():
        temp_json_path.unlink()
        
    log_success(f"Move plans compiled successfully inside Artificial_Intelligence/ folder.")
    return {}

def node_execute_migration(state: OrganizerState) -> Dict[str, Any]:
    log_header("Phase 5: Physical Move & Wikilink Two-Way Repair")
    vault_root = Path(state["vault_root"])

    log_info("Migrating notes, appending YAML metadata, and correcting internal links...")
    execute_process = subprocess.run(
        [sys.executable, str(SCRIPTS_DIR / "organize.py"), "execute"],
        cwd=str(vault_root), capture_output=True, text=True, check=True
    )
    execute_summary = json.loads(execute_process.stdout)
    
    log_success(f"Execution completed. Moved: {execute_summary.get('moved', 0)} files.")
    log_success(f"Wikilinks repaired: {execute_summary.get('links_repaired', 0)} links updated.")
    return {"execute_summary": execute_summary}

def node_review_gate(state: OrganizerState) -> Dict[str, Any]:
    log_header("Phase 6: Interactive Preview & Review Gate")
    execute_summary = state["execute_summary"]
    sandbox_branch = state["sandbox_branch"]

    print(f"\n{Colors.BOLD}{Colors.OKBLUE}📦 VAULT SANDBOX READY FOR HUMAN REVIEW{Colors.ENDC}")
    print("--------------------------------------------------")
    print(f"Your vault has been organized on an isolated sandbox branch: {Colors.BOLD}{sandbox_branch}{Colors.ENDC}")
    print(f"Total Notes Moved: {execute_summary.get('moved', 0)}")
    print(f"Wikilinks Repaired: {execute_summary.get('links_repaired', 0)}")
    print(f"\n{Colors.BOLD}🔍 ACTION REQUIRED: Open Obsidian now!{Colors.ENDC}")
    print("Because we are in Sandbox Mode, you will see the restructured folders, notes,")
    print("and updated links live on your screen inside Obsidian instantly.")
    print("--------------------------------------------------")

    decision = input("How do you want to resolve? (type 'merge' to accept or 'abort' to reject all changes): ").strip().lower()
    if decision in ("merge", "approve", "yes", "y"):
        return {"user_decision": "merge"}
    else:
        return {"user_decision": "abort"}

def node_git_sandbox_resolve(state: OrganizerState) -> Dict[str, Any]:
    vault_root = Path(state["vault_root"])
    original_branch = state["original_branch"]
    sandbox_branch = state["sandbox_branch"]
    stashed = state["stashed"]
    user_decision = state["user_decision"]

    if user_decision == "merge":
        log_info(f"Merging changes into {original_branch}...")
        subprocess.run(["git", "checkout", original_branch], cwd=str(vault_root), check=True)
        subprocess.run(["git", "merge", sandbox_branch], cwd=str(vault_root), check=True)
        subprocess.run(["git", "branch", "-d", sandbox_branch], cwd=str(vault_root), check=True)
        log_success("Merge resolved and sandbox branch deleted cleanly.")
        if stashed:
            log_info("Restoring your stashed active draft changes...")
            subprocess.run(["git", "stash", "pop"], cwd=str(vault_root), check=True)
            log_success("Active draft changes popped back successfully.")
        log_success("WikiLLM Vault Organization session finalized successfully! Workspace synced.")
    else:
        log_warning("Aborting session! Discarding all organized sandbox modifications...")
        subprocess.run(["git", "checkout", original_branch], cwd=str(vault_root), check=True)
        subprocess.run(["git", "reset", "--hard", f"origin/{original_branch}" if stashed else "HEAD"], cwd=str(vault_root), check=False)
        subprocess.run(["git", "branch", "-D", sandbox_branch], cwd=str(vault_root), check=True)
        log_success("All sandbox modifications discarded perfectly.")
        if stashed:
            log_info("Restoring your stashed active draft changes...")
            subprocess.run(["git", "stash", "pop"], cwd=str(vault_root), check=True)
            log_success("Active draft changes popped back successfully.")
        log_warning("WikiLLM Vault Organization session aborted. Workspace restored to original state.")

    return {}

## 🏁 Step 3: Graph Compilation & Execution Loop

Now we construct the `StateGraph`, wire all nodes together sequentially with edges, compile the graph, and trigger stream execution to track node completion live in console!

In [ ]:
def run_master_pipeline():
    log_header("WikiLLM ReAct PARA Vault Organizer Agent Initiated")
    
    # Resolve Vault Root
    from organize import find_vault_root
    try:
        vault_root = find_vault_root()
        log_success(f"Obsidian Vault Root verified: {vault_root}")
    except FileNotFoundError as e:
        log_error(str(e))
        return

    # 1. Compile LangGraph workflow
    log_info("Compiling LangGraph StateGraph workflow...")
    workflow = StateGraph(OrganizerState)

    # Add Nodes
    workflow.add_node("git_sandbox_init", node_git_sandbox_init)
    workflow.add_node("scan_directory", node_scan_directory)
    workflow.add_node("semantic_classify", node_semantic_classify)
    workflow.add_node("generate_move_plan", node_generate_move_plan)
    workflow.add_node("execute_migration", node_execute_migration)
    workflow.add_node("review_gate", node_review_gate)
    workflow.add_node("git_sandbox_resolve", node_git_sandbox_resolve)

    # Set flow edges
    workflow.set_entry_point("git_sandbox_init")
    workflow.add_edge("git_sandbox_init", "scan_directory")
    workflow.add_edge("scan_directory", "semantic_classify")
    workflow.add_edge("semantic_classify", "generate_move_plan")
    workflow.add_edge("generate_move_plan", "execute_migration")
    workflow.add_edge("execute_migration", "review_gate")
    workflow.add_edge("review_gate", "git_sandbox_resolve")
    workflow.add_edge("git_sandbox_resolve", END)

    # Compile
    react_graph = workflow.compile()
    log_success("LangGraph compiled successfully.")

    # Initial State payload
    initial_state = {
        "vault_root": str(vault_root),
        "original_branch": "",
        "sandbox_branch": "",
        "stashed": False,
        "scan_data": {},
        "ai_classified": [],
        "execute_summary": {},
        "user_decision": ""
    }

    # Run stream live
    try:
        for chunk in react_graph.stream(initial_state, stream_mode="updates"):
            for node, update in chunk.items():
                log_success(f"== [LANGGRAPH Node Completed: '{node}'] ==")
    except Exception as e:
        log_error(f"LangGraph execution crashed: {e}")
        log_warning("Executing failure recovery path...")
        try:
            subprocess.run(["git", "checkout", "master"], cwd=str(vault_root), check=False)
            subprocess.run(["git", "stash", "pop"], cwd=str(vault_root), check=False)
        except Exception as recovery_err:
            log_error(f"Shattered recovery path: {recovery_err}")

run_master_pipeline()